# 6 — Network-Spacing Sweep (CAN-only) with Bump-Health Gate

Sweeps the two QAN **accuracy** axes, `sigma` and `offset_magnitude`, with Optuna's
TPE sampler **at several values of the network `spacing`** — the radian resolution that
sets the number of neurons per torus axis (`n ≈ 2π/spacing`). The goal is to see how
re-tiling the torus (coarser/finer) affects (a) the post-transient MADE error and
(b) whether a single coherent activity bump even survives. The search is **CAN-only**:
the arena's torus ground-truth is fed straight into the QAN via `TorchBackend.simulate`,
bypassing the Bingham filter — justified because on the flat arena the floor is
horizontal, so n̂ = ẑ, the rotation step is ≈ identity, and the filter only has to
confirm ẑ (codebase Layer 4 note). Structure mirrors notebook `5_score_flat_run` /
the Experiment-1 sweep.

---
### ⚠️ Read before running — three things the pasted code papered over

**1. `spacing` (this sweep) ≠ `grid_spacing`.** The codebase is explicit that these are
*different unit systems and are not coupled*: `NetworkConfig.spacing` is **radians between
neurons** → it sets `n` (neuron count, connectivity size, bump support);
`ExperimentConfig.grid_spacing` is **metres per full 2π wrap** → it sets
`scale = 2π/grid_spacing` (the metres↔radians gain, i.e. the *physical* grid period).
This notebook sweeps the **network `spacing`** — that is what the pasted `run_sweep`
actually varies (it computes `n = ceil(2π/spacing)` and rebuilds the QAN). If by
"increase the grid spacing" you meant the **physical period `grid_spacing`**, this is the
wrong sweep: that one holds `n` fixed, varies `scale`, and must regenerate `torus_gt`
every step. Tell me which you meant.

**2. Direction is ambiguous, so it's an explicit list.** "Increase the spacing" = *larger*
`spacing` = *fewer* neurons = coarser tiling. The pasted comment `build_connectivity=False`
*"(so the finer spacings construct)"* points the **opposite** way (more neurons, OOM-risk).
`SPACINGS` below spans both sides of the default `0.3`; the preflight table tells you which
direction (if any) actually needs the lazy fix.

**3. `build_connectivity=False` is NOT in the documented constructor.** `arena_2d.py`
builds `Torus3DQAN(spacing, alpha, sigma, b, offset_magnitude)` — no such kwarg — and the
codebase says `CAN3D` builds a **dense** `connectivity_matrix` then deletes it once state
moves to torch. So the lazy fix may not exist yet. The **preflight cell feature-detects it**.
If it's absent, the notebook (a) does *not* pass the kwarg and (b) **skips** any spacing
whose dense matrix would exceed a RAM budget — because a multi-hundred-GB allocation can
trip the OS OOM-killer, which is *not* a catchable `MemoryError`. The pasted `except
(ValueError, RuntimeError, MemoryError)` would not save you from that.

**Decode-granularity confound (interpretation):** MADE error here is
`‖wrapped(decoded − truth)‖ / cumulative_metres`. Coarser spacing → coarser circular-mean
decode → a higher error *floor* that has nothing to do with dynamics. So compare error
across spacings **with the bump-health gate in mind**: a healthy single bump at higher
error may simply be sampled more coarsely.

**Backend caveat (carried over):** this optimizes the **torch blend** backend (`simulate`).
The full pipeline drives the *same* backend via `PathIntegrator`, so §5's validation should
track the sweep; a large gap means a backend/gain mismatch, not seed luck.

Requires: `pip install optuna plotly`.

In [ ]:
import os, sys, gc, inspect
import numpy as np
from scipy import ndimage
import optuna

# Adjust to your layout (mirrors notebook 4_pipeline_integration).
sys.path.insert(0, os.path.abspath("../.."))

from config import RunConfig, NetworkConfig
from experiments.arena_2d import Arena2DExperiment, Arena2DConfig
from experiments.base import BaseExperiment
from network.QAN3D import Torus3DQAN
from network.torch_backend import TorchBackend

# ---- fixed setup ---------------------------------------------------------
ALPHA   = 1.5          # kernel amplitude (feasibility knob, fixed) -> QAN 'alpha'
B       = 0.83         # feedforward drive (feasibility knob, fixed)
T       = 600          # sim length per trial; [200:].mean() leaves ~400 post-transient steps
SEED    = 0
N_TRIALS = 20          # per spacing. 2 axes -> a grid is equally good & more interpretable;
                       # TPE is kept only to match the existing sweeps.

# The axis of interest. Spans both sides of the default 0.3 (n=21) so it works whichever
# direction you meant. Edit this line to set your intended sweep.
#   larger spacing  -> fewer neurons (coarser)      |  smaller spacing -> more neurons (finer)
SPACINGS = [0.6, 0.5, 0.4, 0.3, 0.25]

SIGMA_RANGE  = (0.8, 1.7)
OFFSET_RANGE = (0.15, 0.50)

PENALTY      = 1e3     # clearly worse than any valid MADE error (<< 10)
MAX_DENSE_GB = 8.0     # per-CAN dense-connectivity budget used ONLY when lazy is unsupported
DTYPE_BYTES  = 4       # dense matrix assumed float32 (matches the ~275 GB @ n=64 figure)

setup ok
False 0.05s


## 0 — Preflight: capabilities, memory, and the `torus_gt` invariance check

Three guards that decide whether the sweep below is even valid, run *before* anything
expensive:
- **feature-detect** `build_connectivity` and `TorchBackend.simulate`;
- **memory table** per spacing (dense connectivity is `(n³)² · 4 B` per CAN) so you can see
  which spacings — if any — need the lazy fix or will OOM;
- **`to_phase` invariance**: generating `torus_gt` once and reusing it for every spacing is
  only valid if the manifold's metric map is independent of `n`. We verify it on two cheap
  (coarse) manifolds. If it fails, the "generate once" design is invalid and you'd have to
  regenerate `torus_gt` per spacing.

In [ ]:
sig = inspect.signature(Torus3DQAN.__init__)
SUPPORTS_LAZY = "build_connectivity" in sig.parameters
print("Torus3DQAN.__init__ params :", list(sig.parameters)[1:])
print("supports build_connectivity:", SUPPORTS_LAZY,
      "" if SUPPORTS_LAZY else "  -> kwarg will NOT be passed; dense build + memory cap used")
print("TorchBackend.simulate       :", hasattr(TorchBackend, "simulate"))
if not hasattr(TorchBackend, "simulate"):
    print("  !! simulate() missing — this CAN-only sweep relies on it. Either add it, or drive")
    print("     the QAN via PathIntegrator.run on the ground-truth velocities instead.")

print("\nper-spacing footprint (dense connectivity, ONE CAN, float32):")
print(f"{'spacing':>8} {'n/axis':>7} {'neurons 6n^3':>14} {'dense GB/CAN':>13}  note")
for sp in sorted(set(SPACINGS), reverse=True):
    n  = int(np.ceil(2 * np.pi / sp))
    gb = (n**3)**2 * DTYPE_BYTES / 1e9
    note = "ok (dense fine)" if gb <= 1 else ("dense heavy" if gb <= MAX_DENSE_GB
                                              else "NEEDS lazy / will OOM")
    print(f"{sp:>8.3f} {n:>7d} {6*n**3:>14,} {gb:>13,.2f}  {note}")

# --- to_phase invariance: justifies one shared torus_gt across all spacings ---
def _metric_at(sp):
    kw = dict(spacing=sp, sigma=1.4, offset_magnitude=0.25, alpha=ALPHA, b=B)
    if SUPPORTS_LAZY:
        kw["build_connectivity"] = False
    q = Torus3DQAN(**kw)
    m = q.manifold.metric
    del q; gc.collect()
    return m

try:
    test = np.array([[1., 0.], [0., 1.], [1., 1.], [2., 0.5]])
    pa = _metric_at(0.7).to_phase(test)   # n=9
    pb = _metric_at(0.5).to_phase(test)   # n=13
    if np.allclose(pa, pb, atol=1e-9):
        print("\nto_phase is spacing-invariant — one torus_gt is valid for all spacings.")
    else:
        print("\n!! to_phase DIFFERS across spacing — regenerate torus_gt per spacing;")
        print("   the 'generate once' design below is INVALID. max|diff| =", np.abs(pa - pb).max())
except Exception as e:
    print("\n(could not run to_phase invariance check:", type(e).__name__, str(e), ")")

## 1 — Trajectory (generated once, via the arena)

`torus_gt` depends only on the manifold metric and the experiment config (`grid_spacing` →
`scale`), **not** on the network `spacing`/`sigma`/`offset` — confirmed by the invariance
check above. So we generate it **once** at the default network spacing (n=21, ~0.3 GB dense —
cheap) and feed the *same* `world_pos`/`torus_gt` to every trial at every spacing. That makes
differences across trials purely parametric. `grid_spacing` is held at its default, so the
metres↔radians gain is identical across the sweep.

In [ ]:
base = Arena2DExperiment(
    RunConfig(network=NetworkConfig(), experiment=Arena2DConfig(n_steps=T, seed=SEED)),
    record=False,
)
world_pos, v_body_seq, torus_gt = base.generate_trajectory()
print("torus_gt:", torus_gt.shape, " world_pos:", world_pos.shape)
del base; gc.collect()   # the reference QAN is no longer needed

## 2 — Bump-health metric, calibration, objective, and `run_sweep`

### What changed vs the pasted version (and why)

- **`bump_metrics` no longer trusts `ceil(2π/spacing)` for the reshape.** It derives the cube
  side from the state itself (`round(N**(1/3))`) and **raises** if `N` is not a perfect cube.
  The pasted `reshape(n, n, n)` would silently misinterpret the activity if the QAN's true
  per-axis count differed from `ceil(2π/spacing)` (e.g. if it rounds differently). The
  objective passes the QAN's *actual* `n` and warns once if it disagrees with the formula.
- **The gate is scale-robust.** The pasted gate (`n_peaks != 1`) accepts a *flat/smeared*
  field: when the field is nearly uniform, `max ≈ mean`, so the half-max blob covers the whole
  torus and `ndimage.label` returns exactly **one** component — a dead-but-uniform network
  passes. We add `coverage` = fraction of voxels above half-max, which is ≈ a geometric ratio
  (bump width / 2π) and therefore roughly **independent of `n`**, unlike `peak_contrast` and
  `active_fraction` (both drift with `n`: as the torus is sampled more finely the same physical
  bump occupies a smaller *fraction* of `n³`). The hard gate is **`n_peaks == 1` and
  `coverage ≤ coverage_max`**; `peak_contrast`/`active_fraction` are logged but not gated.
- **The gate threshold is *calibrated from a known-good run*, not guessed.** We measure
  `coverage` on the validated default config (notebook 3 calls this the clean CAN upper bound)
  and set `coverage_max` relative to it.
- **`world_pos`/`torus_gt` are passed in, not read from globals**, and the dense-memory check
  runs **once per spacing** (skipping infeasible spacings) rather than letting `n_trials`
  allocations each crash. `TypeError` is also caught, in case `build_connectivity` slipped past
  the feature check.

In [ ]:
def bump_metrics(S_tot, n=None):
    """
    Diagnostics on the relu'd bump activity. S_tot: 1-D array of N = n**3 values
    (one CAN's worth, i.e. backend.S.mean(dim=0)).

      n_peaks         : connected blobs above half-max after rolling the global peak to
                        centre (1 = healthy, 0 = dead, >1 = split).
      coverage        : fraction of voxels above half-max. ~scale-invariant; large => smeared.
      peak_contrast   : max / mean of positive activity (drifts up with n; logged only).
      active_fraction : participation ratio / N  (effective support; drifts down with n; logged).
      status          : 'ok' | 'dead' | 'split' | 'smeared'.
    """
    r = np.maximum(np.asarray(S_tot, dtype=float).ravel(), 0.0)
    N = r.size
    if n is None:
        n = int(round(N ** (1.0 / 3.0)))
    if n ** 3 != N:
        raise ValueError(
            f"bump_metrics: S_tot has {N} entries, not a cube of n={n} (n^3={n**3}). "
            "Pass the QAN's true per-axis size; do not assume ceil(2*pi/spacing)."
        )
    total = r.sum()
    if total <= 0.0:
        return dict(n_peaks=0, coverage=0.0, peak_contrast=0.0,
                    active_fraction=1.0, status="dead")

    pr              = total ** 2 / (np.square(r).sum() + 1e-12)     # participation ratio
    active_fraction = pr / N
    peak_contrast   = r.max() / (r.mean() + 1e-12)

    cube = r.reshape(n, n, n)
    thr  = 0.5 * cube.max()
    coverage = float((cube >= thr).mean())

    # de-seam the main bump (roll its peak to centre) before counting blobs on the 3-torus
    pk = np.unravel_index(cube.argmax(), cube.shape)
    shifted = np.roll(cube, [s // 2 - p for s, p in zip(cube.shape, pk)], axis=(0, 1, 2))
    _, n_blobs = ndimage.label(shifted >= thr)
    n_peaks = int(n_blobs)

    status = ("dead" if n_peaks == 0 else
              "split" if n_peaks > 1 else
              "smeared" if coverage > 0.30 else "ok")
    return dict(n_peaks=n_peaks, coverage=coverage, peak_contrast=float(peak_contrast),
                active_fraction=float(active_fraction), status=status)

In [ ]:
# --- calibrate the gate from the validated default config (the known-good bump) ---
kw = dict(spacing=NetworkConfig.spacing, sigma=NetworkConfig.sigma,
          offset_magnitude=NetworkConfig.offset_magnitude,
          alpha=NetworkConfig.kernel_alpha, b=NetworkConfig.b)
if SUPPORTS_LAZY:
    kw["build_connectivity"] = False

ref_qan = Torus3DQAN(**kw)
ref_backend = TorchBackend(qan=ref_qan)
_ = ref_backend.simulate(torus_gt)
ref_S = ref_backend.S.mean(dim=0).detach().cpu().numpy().ravel()
ref_n = int(round(ref_S.size ** (1.0 / 3.0)))
ref_bm = bump_metrics(ref_S, ref_n)
print(f"reference bump (default cfg, spacing={NetworkConfig.spacing}, n={ref_n}): {ref_bm}")
if ref_bm["n_peaks"] != 1:
    print("  !! default config is itself not a single bump — calibration is unreliable; "
          "fix ALPHA/B/sigma/offset before sweeping.")

GATE = dict(coverage_max=max(0.20, 3.0 * ref_bm["coverage"]))   # reference-aware, floored
print("GATE:", GATE, " (a trial is 'healthy' iff n_peaks==1 and coverage<=coverage_max)")
del ref_qan, ref_backend; gc.collect()

In [ ]:
def make_objective(spacing, world_pos, torus_gt, sigma_range, offset_range, gate):
    warned = {"n": False}

    def objective(trial):
        sg  = trial.suggest_float("sigma",            *sigma_range)
        off = trial.suggest_float("offset_magnitude", *offset_range)
        qan = backend = None
        try:
            kwargs = dict(spacing=spacing, sigma=sg, offset_magnitude=off, alpha=ALPHA, b=B)
            if SUPPORTS_LAZY:
                kwargs["build_connectivity"] = False
            qan = Torus3DQAN(**kwargs)
            backend = TorchBackend(qan=qan)
            decoded = backend.simulate(torus_gt)

            err = float(BaseExperiment._made_metric(decoded, torus_gt, world_pos=world_pos)[200:].mean())

            # bump diagnostics on the final state — the same S_tot the decoder reads
            S_tot   = backend.S.mean(dim=0).detach().cpu().numpy().ravel()
            n_state = int(round(S_tot.size ** (1.0 / 3.0)))
            if not warned["n"] and n_state != int(np.ceil(2 * np.pi / spacing)):
                print(f"  note: QAN per-axis n={n_state} != ceil(2*pi/spacing)="
                      f"{int(np.ceil(2*np.pi/spacing))}; using the QAN's n.")
                warned["n"] = True
            bm = bump_metrics(S_tot, n_state)
            for k, v in bm.items():
                trial.set_user_attr(k, v)

            healthy = (bm["n_peaks"] == 1 and bm["coverage"] <= gate["coverage_max"])
            trial.set_user_attr("healthy", bool(healthy))
            if not healthy:           # dead / split / smeared: the MADE score is meaningless
                return PENALTY
            return err if np.isfinite(err) else PENALTY
        except (TypeError, ValueError, RuntimeError, MemoryError) as e:
            print(f"  FAIL spacing={spacing:.3f} sigma={sg:.3f} off={off:.3f}: {type(e).__name__}: {e}")
            return PENALTY
        finally:
            del qan, backend
            gc.collect()
    return objective


def run_sweep(spacing, world_pos, torus_gt, n_trials=N_TRIALS, seed=SEED,
              sigma_range=SIGMA_RANGE, offset_range=OFFSET_RANGE,
              gate=None, max_dense_gb=MAX_DENSE_GB):
    """CAN-only sigma/offset sweep at a fixed network spacing, with a bump-health gate."""
    n  = int(np.ceil(2 * np.pi / spacing))
    gb = (n**3)**2 * DTYPE_BYTES / 1e9
    feasible = SUPPORTS_LAZY or gb <= max_dense_gb
    print(f"--- spacing={spacing:.3f} | n~{n}/axis | {6*n**3:,} neurons | "
          f"dense~{gb:,.2f} GB/CAN | lazy={SUPPORTS_LAZY} | feasible={feasible} ---")
    if not feasible:
        print(f"  SKIP: dense build {gb:,.1f} GB > budget {max_dense_gb} GB and no lazy path. "
              "Add build_connectivity to Torus3DQAN/CAN3D, or raise max_dense_gb if you truly "
              "have the RAM.")
        return None

    gate = gate or GATE
    study = optuna.create_study(direction="minimize",
                                sampler=optuna.samplers.TPESampler(seed=seed))
    study.optimize(make_objective(spacing, world_pos, torus_gt, sigma_range, offset_range, gate),
                   n_trials=n_trials)
    n_ok = sum(t.user_attrs.get("healthy", False) for t in study.trials)
    if n_ok == 0:
        print(f"  no healthy single-bump trial (all dead/split/smeared) — best value is the "
              f"penalty floor {PENALTY:.0f}.")
    else:
        print(f"  best {study.best_params} -> {study.best_value:.5f}   ({n_ok}/{n_trials} healthy)")
    return study

## 3 — Run the sweep across spacings

One study per spacing; infeasible spacings are skipped (not crashed). Results are collected
into a tidy frame: best (σ, δ), the gated error, the healthy-trial count, and the bump
diagnostics of the winning trial.

In [ ]:
import pandas as pd

studies, rows = {}, []
for sp in SPACINGS:
    st = run_sweep(sp, world_pos, torus_gt)
    studies[sp] = st
    n = int(np.ceil(2 * np.pi / sp))
    if st is None:
        rows.append(dict(spacing=sp, n=n, feasible=False, n_healthy=0))
        continue
    bt = st.best_trial
    rows.append(dict(
        spacing=sp, n=n, feasible=True,
        n_healthy=sum(t.user_attrs.get("healthy", False) for t in st.trials),
        best_sigma=st.best_params.get("sigma"),
        best_offset=st.best_params.get("offset_magnitude"),
        best_error=st.best_value,
        coverage=bt.user_attrs.get("coverage"),
        peak_contrast=bt.user_attrs.get("peak_contrast"),
        n_peaks=bt.user_attrs.get("n_peaks"),
    ))
    gc.collect()

summary = pd.DataFrame(rows).sort_values("spacing").reset_index(drop=True)
summary

## 4 — Inspect

`plot_contour` per spacing shows the σ–δ response surface; here we add the **cross-spacing**
view that this notebook exists for. Remember the decode-granularity confound: a rising error
floor at larger spacing is partly just coarser decoding, so read it together with `n_healthy`
and `coverage`, not in isolation.

In [ ]:
import optuna.visualization as vis

# per-spacing surface for the finest feasible spacing that produced healthy trials
feas = [sp for sp in SPACINGS if studies.get(sp) is not None
        and any(t.user_attrs.get("healthy", False) for t in studies[sp].trials)]
if feas:
    sp_show = min(feas)
    print(f"contour / importances for spacing={sp_show:.3f}")
    vis.plot_contour(studies[sp_show], params=["sigma", "offset_magnitude"]).show()
    vis.plot_param_importances(studies[sp_show]).show()
else:
    print("no spacing produced a healthy trial — nothing to contour.")

In [ ]:
import matplotlib.pyplot as plt

ok = summary[(summary["feasible"]) & (summary["n_healthy"] > 0)].sort_values("spacing")
if len(ok):
    fig, ax1 = plt.subplots(figsize=(6.5, 4.2))
    ax1.plot(ok["spacing"], ok["best_error"], "o-", color="steelblue")
    ax1.set_xlabel("network spacing (rad / neuron)")
    ax1.set_ylabel("best post-transient MADE error", color="steelblue")
    ax1.tick_params(axis="y", labelcolor="steelblue")
    ax1.grid(True, alpha=0.3)

    ax2 = ax1.twinx()
    ax2.plot(ok["spacing"], 6 * ok["n"] ** 3, "s--", color="indianred", alpha=0.6)
    ax2.set_ylabel("total neurons (6 n^3, log)", color="indianred")
    ax2.set_yscale("log")
    ax2.tick_params(axis="y", labelcolor="indianred")

    ax1.set_title("MADE error vs network spacing (healthy single-bump trials only)")
    fig.tight_layout(); plt.show()
else:
    print("nothing healthy to plot.")

## 5 — Validate the winner in the *full* pipeline

Re-introduce the Bingham filter and the metres↔radians mapping by running the real
`Arena2DExperiment` at the overall-best `(spacing, σ, δ)` across a few seeds. The winning
network `spacing` goes into `NetworkConfig(spacing=...)`; `grid_spacing` stays at its default,
so this isolates the network-resolution change exactly as the CAN-only sweep did. On the flat
arena the filter is near-idle, so a large gap between these numbers and the sweep's best value
usually means the pipeline drives a *different* QAN backend than `simulate` (see the backend
caveat at the top).

In [ ]:
ok = summary[(summary["feasible"]) & (summary["n_healthy"] > 0)].sort_values("best_error")
if len(ok) == 0:
    print("No feasible + healthy spacing to validate.")
else:
    win = ok.iloc[0]
    print(f"overall winner: spacing={win['spacing']:.3f}  sigma={win['best_sigma']:.3f}  "
          f"offset={win['best_offset']:.3f}  CAN-only err={win['best_error']:.5f}")
    g_vec = np.array([0., 0., -9.81])
    for seed in range(3):
        cfg = RunConfig(
            network=NetworkConfig(spacing=float(win["spacing"]),
                                  sigma=float(win["best_sigma"]),
                                  offset_magnitude=float(win["best_offset"])),  # alpha/b = defaults
            experiment=Arena2DConfig(n_steps=5000, seed=seed),
        )
        res = Arena2DExperiment(cfg, record=False).run_experiment(g=g_vec)
        print(f"  seed {seed}: full-pipeline mean_norm_error = {res.mean_norm_error:.4f}")
        gc.collect()

## 6 — Scaling notes

- **Which direction needs the lazy fix?** Read it off the preflight table, not intuition.
  *Coarser* spacing (the literal reading of "increase the spacing") makes `n` *smaller* and the
  dense matrix *cheaper* — `build_connectivity=False` is then unnecessary, and the real risk is
  the **bump dying or fragmenting** as the torus is under-sampled, which is exactly what the
  gate catches. *Finer* spacing (more neurons) is where the dense `(n³)²` connectivity OOMs and
  the lazy path is mandatory.
- **n = 64 / rented GPU:** only after `connectivity_matrix` is built lazily (or skipped) in the
  MADE `CAN`; otherwise construction OOMs *before* torch runs (and the OS OOM-killer is not a
  catchable `MemoryError`).
- **Two axes:** with just σ and δ a grid is as good as TPE and more interpretable; TPE earns its
  keep once you add the filter axes (`kappa`, `bingham_decay`) and seed-averaging. The value
  *here* is the spacing dimension, not the per-spacing optimizer.
- **Pruning** (ASHA/median) needs intermediate values: have `simulate` call
  `trial.report(running_error, t)` at checkpoints and
  `if trial.should_prune(): raise optuna.TrialPruned()`.
- **Calibrate the gate against notebook 3** (the clean CAN upper bound) if `coverage_max` ever
  rejects a bump you can see is healthy in the activity plots, or accepts one you can see is not.